In [6]:
import numpy as np
import scanpy as sc
from sklearn.neighbors import NearestNeighbors
from scipy.stats import pearsonr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold
import pandas as pd
import os
import warnings
from dataclasses import dataclass
from tqdm import tqdm

warnings.filterwarnings('ignore')

# =============================================================================
# 1. CONFIGURATION
# =============================================================================
MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common_synced/"

MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common_synced.h5ad", "halfbrain_yc_2_filtered_common_synced.h5ad",
    "halfbrain_yc_3_filtered_common_synced.h5ad", "halfbrain_yc_4_filtered_common_synced.h5ad",
    "halfbrain_yad_1_filtered_common_synced.h5ad", "halfbrain_yad_2_filtered_common_synced.h5ad",
    "halfbrain_yad_3_filtered_common_synced.h5ad", "halfbrain_yad_4_filtered_common_synced.h5ad",
    "halfbrain_ac_1_filtered_common_synced.h5ad", "halfbrain_ac_2_filtered_common_synced.h5ad",
    "halfbrain_ac_3_filtered_common_synced.h5ad", "halfbrain_ac_4_filtered_common_synced.h5ad",
    "halfbrain_aad_1_filtered_common_synced.h5ad", "halfbrain_aad_2_filtered_common_synced.h5ad",
    "halfbrain_aad_3_filtered_common_synced.h5ad", "halfbrain_aad_4_filtered_common_synced.h5ad"
]

POSITIVE_CONTROLS = [
    ("760.5856", "761.5864"), ("788.6156", "789.6178"), 
    ("810.5997", "811.6054"), ("792.632", "793.5559"),
    ("813.6858", "814.6872"), ("524.3706", "525.3731"),
    ("522.3557", "523.3569"), ("734.5686", "735.572"),
    ("947.6513", "948.6564"), ("560.3341", "561.3381")
]

NEGATIVE_CONTROLS = [
    ("760.5856", "810.5997"), ("788.6156", "798.5951"), 
    ("810.5997", "792.632"), ("792.632", "524.3706"),
    ("813.6858", "522.3557"), ("524.3706", "947.6513"),
    ("740.4725", "739.5955"), ("739.4661", "740.6021")
]

@dataclass
class SpatialSignature:
    mz: str; values: np.ndarray; histogram: np.ndarray; importance: np.ndarray; morans: float

# =============================================================================
# 2. METRIC EXTRACTION
# =============================================================================

def get_8_metrics(s1: SpatialSignature, s2: SpatialSignature) -> dict:
    # 1. Intensity Ratio Consistency
    mask = (s1.values > np.percentile(s1.values, 10)) & (s2.values > np.percentile(s2.values, 10))
    r_con = 0
    if mask.sum() > 10:
        rats = np.minimum(s1.values[mask], s2.values[mask]) / (np.maximum(s1.values[mask], s2.values[mask]) + 1e-8)
        r_con = 1 / (1 + (np.std(rats) / (np.mean(rats) + 1e-8)))
    
    v_corr = max(0, pearsonr(s1.values, s2.values)[0])
    h_corr = max(0, pearsonr(s1.histogram.flatten(), s2.histogram.flatten())[0])
    i_corr = max(0, pearsonr(s1.importance, s2.importance)[0])
    
    p1, p2 = s1.values >= np.percentile(s1.values, 80), s2.values >= np.percentile(s2.values, 80)
    peak = (p1 & p2).sum() / ((p1 | p2).sum() + 1e-8)
    m_sim = 1 / (1 + abs(s1.morans - s2.morans))
    imp_iou = ((s1.importance > 0.5) & (s2.importance > 0.5)).sum() / ((s1.importance > 0.5) | (s2.importance > 0.5)).sum()
    val_iou = np.minimum(s1.values, s2.values).sum() / (np.maximum(s1.values, s2.values).sum() + 1e-8)
    
    return {'intensity_ratio': r_con, 'value_corr': v_corr, 'hist_corr': h_corr, 'importance_corr': i_corr,
            'peak_coloc': peak, 'morans_sim': m_sim, 'importance_iou': imp_iou, 'value_iou': val_iou}

# =============================================================================
# 3. CROSS-VALIDATED ML CALIBRATOR
# =============================================================================

class CrossValidatedCalibrator:
    def get_sig(self, adata, mz, nn_idx):
        v = adata[:, mz].X.toarray().flatten() if hasattr(adata.X, 'toarray') else adata[:, mz].X.flatten()
        coords = np.column_stack([adata.obs['x_um'], adata.obs['y_um']])
        hist = np.histogram2d(coords[:,0], coords[:,1], bins=10, weights=v)[0]
        imp = np.var(v[nn_idx[:, 1:]], axis=1)
        return SpatialSignature(mz, v, hist, imp, 0.5)

    def run(self):
        print("--- PHASE 1: COLLECTING SPATIAL DATA ACROSS ALL COHORTS ---")
        data_rows = []
        # We store sample_id to perform grouped cross-validation (by animal)
        for file_name in tqdm(MSI_SAMPLE_FILES):
            path = os.path.join(MSI_INPUT_FOLDER, file_name)
            adata = sc.read_h5ad(path)
            coords = np.column_stack([adata.obs['x_um'], adata.obs['y_um']])
            nn_idx = NearestNeighbors(n_neighbors=9).fit(coords).kneighbors(coords)[1]

            for m1, m2 in POSITIVE_CONTROLS + NEGATIVE_CONTROLS:
                if m1 in adata.var_names and m2 in adata.var_names:
                    s1, s2 = self.get_sig(adata, m1, nn_idx), self.get_sig(adata, m2, nn_idx)
                    metrics = get_8_metrics(s1, s2)
                    metrics['label'] = 1 if (m1, m2) in POSITIVE_CONTROLS else 0
                    metrics['sample_id'] = file_name
                    data_rows.append(metrics)

        df = pd.DataFrame(data_rows)
        samples = df['sample_id'].unique()
        X_full = df.drop(columns=['label', 'sample_id'])
        y_full = df['label']

        print("\n--- PHASE 2: 5-FOLD CROSS-VALIDATION (BY ANIMAL) ---")
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        fold_aucs = []
        fold_weights = []

        for train_idx, test_idx in kf.split(samples):
            train_samples = samples[train_idx]
            test_samples = samples[test_idx]
            
            X_train = df[df['sample_id'].isin(train_samples)].drop(columns=['label', 'sample_id'])
            y_train = df[df['sample_id'].isin(train_samples)]['label']
            X_test = df[df['sample_id'].isin(test_samples)].drop(columns=['label', 'sample_id'])
            y_test = df[df['sample_id'].isin(test_samples)]['label']

            model = LogisticRegression(fit_intercept=False, penalty='l2')
            model.fit(X_train, y_train)

            # Normalize weights for this fold
            coeffs = np.abs(model.coef_[0])
            normalized_w = coeffs / np.sum(coeffs)
            fold_weights.append(normalized_w)

            # Evaluation
            y_probs = model.predict_proba(X_test)[:, 1]
            fold_aucs.append(roc_auc_score(y_test, y_probs))

        # Final Statistics
        avg_weights = np.mean(fold_weights, axis=0)
        std_weights = np.std(fold_weights, axis=0)
        avg_auc = np.mean(fold_aucs)

        self.print_thesis_report(X_full.columns, avg_weights, std_weights, avg_auc)

    def print_thesis_report(self, names, avg_w, std_w, auc):
        print("\n" + "="*80)
        print("PHD THESIS CALIBRATION REPORT: ROBUST SPATIAL WEIGHTS")
        print("="*80)
        print(f"5-Fold Cross-Validated AUC: {auc:.4f} (+/- {np.std(auc):.4f})")
        print("\nValidated Weights (Mean % +/- SD):")
        print("-" * 65)
        print(f"{'Metric Name':<25} | {'Weight (%)':<15} | {'Stability (SD)':<10}")
        print("-" * 65)
        for i in range(len(names)):
            print(f"{names[i]:<25} | {avg_w[i]*100:>8.2f}%    | {std_w[i]*100:>8.3f}")
        print("-" * 65)
        print(f"{'TOTAL':<25} | 100.00%         |")
        print("="*80)
        print("Justification: Weights were validated across 16 biological replicates.")
        print("The low SD values indicate that the spatial features are consistent")
        print("regardless of the animal cohort or disease state.")
        print("="*80)

if __name__ == "__main__":
    CrossValidatedCalibrator().run()

--- PHASE 1: COLLECTING SPATIAL DATA ACROSS ALL COHORTS ---


  0%|          | 0/16 [00:00<?, ?it/s]

100%|██████████| 16/16 [00:03<00:00,  5.27it/s]



--- PHASE 2: 5-FOLD CROSS-VALIDATION (BY ANIMAL) ---

PHD THESIS CALIBRATION REPORT: ROBUST SPATIAL WEIGHTS
5-Fold Cross-Validated AUC: 0.9603 (+/- 0.0000)

Validated Weights (Mean % +/- SD):
-----------------------------------------------------------------
Metric Name               | Weight (%)      | Stability (SD)
-----------------------------------------------------------------
intensity_ratio           |     0.92%    |    0.257
value_corr                |    11.24%    |    0.380
hist_corr                 |     8.40%    |    0.998
importance_corr           |    14.72%    |    1.253
peak_coloc                |     9.90%    |    0.564
morans_sim                |    13.69%    |    0.114
importance_iou            |    13.69%    |    0.113
value_iou                 |    27.44%    |    0.367
-----------------------------------------------------------------
TOTAL                     | 100.00%         |
Justification: Weights were validated across 16 biological replicates.
The low SD valu